# Koddan Kasa · Levha 04 · 4.096 kopya, 64 kayganlık

Robot köpekler yürümeyi simülasyonda öğreniyor ve her kopya farklı kayganlıkta bir zeminde yürüyor. ETH'nin açık kodu [`legged_gym`](https://github.com/leggedrobotics/legged_gym/blob/master/legged_gym/envs/base/legged_robot_config.py) (Unitree'nin [`unitree_rl_gym`](https://github.com/unitreerobotics/unitree_rl_gym)'i bunun üstüne kurulu, Go2 bu satırları değiştirmiyor):

```python
num_envs = 4096                 # 4.096 kopya
randomize_friction = True
friction_range = [0.5, 1.25]    # sürtünme katsayısı μ
num_buckets = 64                # (legged_robot.py) 64 farklı μ, kopyalara rastgele dağıtılıyor
```

**Kural (Coulomb):** bir blok eğik düzlemde `tan θ = μ` olduğunda kaymaya başlar. Ağırlık iki tarafta da var, sadeleşir: kayma açısı = arctan μ, **kütleden bağımsız**.

**Deney:** Unitree Go2'nin fizik modeli (Google DeepMind MuJoCo Menagerie) duruşunu eklem PD'siyle tutuyor (yapay zekâ politikası yok). Zemin saniyede 1° eğiliyor; köpeğin burnu yokuş yukarı. Ne zaman kayıyor, ne zaman devriliyor?

- Kazançlar **Kp 60 / Kd 5**: Unitree'nin gerçek Go2 için kalkma örneği ([`go2_stand_example.cpp`](https://github.com/unitreerobotics/unitree_sdk2/blob/main/example/go2/go2_stand_example.cpp)). Eğitimdeki 20 / 0,5 politikasız tutulunca arka bacaklar ~10°'de katlanıyor (aşağıda dene).
- Zemini döndürmek yerine **yerçekimini zemine göre döndürüyoruz**: fiziği aynı (eğim çok yavaş), zemin ayakların altında hiç kıpırdamıyor.

## 1 · Kurulum (MuJoCo + sadece Go2 modeli)

In [ ]:
!pip -q install mujoco
# Birebir aynı sayılar için: !pip -q install mujoco==3.17.0
![ -d mujoco_menagerie ] || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/google-deepmind/mujoco_menagerie
!cd mujoco_menagerie && git sparse-checkout set unitree_go2 && ls unitree_go2

## 2 · Deney: yavaşça eğilen zemin

Protokol (`src/v6/kk04v3/experiment.py` ile aynı):
- Köpek `default_joint_angles` duruşunda 2 sn düz zeminde bekler, sonra zemin 1°/sn eğilir.
- Ayakların sürtünmesi = zeminin sürtünmesi = **μ** (Menagerie'de ayaklar `priority=1`: temasın μ'sü ayağınki).
- Her adımda ayak başına normal kuvvet **N** ve yanal kuvvet **T** okunur. `|T|/N` μ'ye dayanınca o ayak sürtünme sınırındadır.
- **Kaydı:** aşağıdaki (arka) ayakların ortalaması > 2 cm/sn ve 1 sn içinde ≥ 3 cm daha gidiyor.
- **Devrildi:** iki yukarı (ön) ayak ≥ 0,1 sn yerden kesik.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
import mujoco
import numpy as np

GO2 = "mujoco_menagerie/unitree_go2/go2.xml"
STAND = np.array([0.1, 0.8, -1.5, -0.1, 0.8, -1.5, 0.1, 1.0, -1.5, -0.1, 1.0, -1.5])  # go2_config default_joint_angles
FEET = ["FL", "FR", "RL", "RR"]
SETTLE, V_TH, SLIDE_CM, LOG_HZ, POST = 2.0, 2.0, 3.0, 120, 2.5
ORIENTS = {"up": ((-1.0, 0.0), ("FL", "FR")), "down": ((1.0, 0.0), ("RL", "RR")), "side": ((0.0, -1.0), ("FL", "RL"))}
P, NN, TN, OT = 21, 25, 29, 33  # log sütunları: ayak konumu, N, |T|/N, başka temas


def build(mu, load=0.0):
    spec = mujoco.MjSpec.from_file(GO2)
    for k in list(spec.keys):
        spec.delete(k)
    spec.worldbody.add_geom(name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=[0, 0, 0.05], friction=[mu, 0.005, 0.0001])
    m = spec.compile()
    for f in FEET:
        m.geom_friction[m.geom(f).id, 0] = mu
    if load:
        m.body_mass[m.body("base").id] += load
    return m


def events(a, mu, up_feet, final=True):
    t, th = a[:, 0], a[:, 1]
    tilt = th > 0
    up = [FEET.index(f) for f in up_feet]
    dn = [k for k in range(4) if k not in up]
    pos, N, R = a[:, P:P + 4], a[:, NN:NN + 4], a[:, TN:TN + 4]
    s0 = int(np.argmax(tilt)) if tilt.any() else len(a) - 1
    up_mv = pos[:, up].mean(1) - pos[s0, up].mean()
    idx = np.where(tilt & (up_mv >= 1.0) & (N[:, up].min(1) > 0))[0]
    up_slip = int(idx[0]) if len(idx) else None
    idx = np.where(tilt & (R[:, up].min(1) >= 0.97 * mu))[0]
    up_sat = int(idx[0]) if len(idx) else None
    off = tilt & (N[:, up].max(1) <= 0)
    n = int(0.1 * LOG_HZ)
    tip = next((int(j) for j in np.where(off)[0] if j + n <= len(a) and off[j:j + n].all()), None)
    dp = pos[:, dn].mean(1)
    k = int(0.125 * LOG_HZ)
    v = np.zeros(len(a))
    if len(a) > 2 * k:
        v[k:-k] = (dp[2 * k:] - dp[:-2 * k]) / (t[2 * k:] - t[:-2 * k])
    slide = None
    for j in np.where(tilt & (v > V_TH))[0]:
        later = np.where(t <= t[j] + 1.0)[0][-1]
        if not final or t[-1] >= t[j] + 1.0:
            if dp[later] - dp[j] >= SLIDE_CM:
                slide = int(j)
                break
    lost = min([x for x in (tip, slide) if x is not None], default=None)
    deg = lambda j: None if j is None else round(float(th[j]), 2)
    other = tilt & (a[:, OT] > 0)
    return dict(event=None if lost is None else ("tip" if lost == tip else "slide"), lost_deg=deg(lost),
                lost_t=None if lost is None else round(float(t[lost]), 3), tip_deg=deg(tip), slide_deg=deg(slide),
                up_slip_deg=deg(up_slip), up_sat_deg=deg(up_sat), touch_deg=deg(int(np.argmax(other))) if other.any() else None)


def run(mu, load=0.0, kp=60.0, kd=5.0, rate=1.0, max_deg=70.0, orient="up"):
    m = build(mu, load)
    d = mujoco.MjData(m)
    d.qpos[7:] = STAND
    d.qpos[2] = 0.40
    mujoco.mj_forward(m, d)
    lo, hi = m.actuator_ctrlrange[:, 0], m.actuator_ctrlrange[:, 1]
    g0 = 9.81
    W = float(m.body_subtreemass[0]) * g0
    (ux, uy), up_feet = ORIENTS[orient]
    fid = {m.geom(f).id: k for k, f in enumerate(FEET)}
    floor = m.geom("floor").id
    every = max(1, int(round(1 / LOG_HZ / m.opt.timestep)))
    log, i, f6, t_lost = [], 0, np.zeros(6), None
    while True:
        th = max(0.0, (d.time - SETTLE) * rate)
        if th > max_deg or (t_lost is not None and d.time > t_lost + POST):
            break
        r = np.radians(th)
        m.opt.gravity[:] = [ux * g0 * np.sin(r), uy * g0 * np.sin(r), -g0 * np.cos(r)]  # zemini eğmek = yerçekimini döndürmek
        d.ctrl[:] = np.clip(kp * (STAND - d.qpos[7:]) - kd * d.qvel[6:], lo, hi)        # eklem PD, motor sınırında kırpılır
        mujoco.mj_step(m, d)
        if i % every == 0:
            N, T, other = np.zeros(4), np.zeros(4), 0
            for c in range(d.ncon):
                con = d.contact[c]
                if floor not in (con.geom1, con.geom2):
                    continue
                o = con.geom2 if con.geom1 == floor else con.geom1
                if o in fid:
                    mujoco.mj_contactForce(m, d, c, f6)
                    N[fid[o]] += f6[0]
                    T[fid[o]] += np.hypot(f6[1], f6[2])
                else:
                    other = 1
            pos = [float(d.geom_xpos[m.geom(f).id][0] * ux + d.geom_xpos[m.geom(f).id][1] * uy) * 100 for f in FEET]
            log.append([round(d.time, 4), round(th, 3), *[round(float(x), 5) for x in d.qpos],
                        *[round(x, 3) for x in pos], *[round(float(n / W * 100), 2) for n in N],
                        *[round(float(T[k] / N[k]), 3) if N[k] > 1e-3 else 0.0 for k in range(4)], other])
            if t_lost is None and th > 0 and len(log) % 12 == 0:
                if events(np.array(log), mu, up_feet, final=False)["lost_deg"] is not None:
                    t_lost = d.time
        i += 1
    a = np.array(log)
    res = dict(mu=mu, theory_deg=round(float(np.degrees(np.arctan(mu))), 2), load_kg=load, kp=kp, kd=kd, **events(a, mu, up_feet))
    return res, a

print("hazır")

## 3 · ✏️ BURAYI DEĞİŞTİR

Videodaki sayılar: `MU = 0.1` (buz gibi) → 6°'de kayıyor · `MU = 0.5` → 25,6°'de devriliyor · `MU = 1.25` → 27,5°'de devriliyor. Dene: `LOAD_KG = 5`, `ORIENT = "side"` (yan durursa), `KP, KD = 20, 0.5` (eğitimdeki sayılar, politikasız).

In [ ]:
# ===================== BURAYI DEĞİŞTİR =====================
MU = 0.5         # sürtünme katsayısı (eğitim: 0,5 – 1,25 · buz ≈ 0,1)
LOAD_KG = 0      # sırttaki ek yük [kg]
KP, KD = 60, 5   # eklem PD kazançları
ORIENT = "up"    # "up" burun yukarı · "down" burun aşağı · "side" yan
# ===========================================================

s, a = run(MU, LOAD_KG, KP, KD, orient=ORIENT)
olay = {"slide": "KAYDI", "tip": "DEVRİLDİ", None: "70°'ye kadar dayandı"}[s["event"]]
print(f"μ = {MU}   yük = {LOAD_KG} kg   Kp/Kd = {KP}/{KD}   yön = {ORIENT}")
print(f"blok teorisi arctan μ      : {s['theory_deg']}°")
print(f"köpek                      : {olay} · {s['lost_deg']}°")
print(f"yukarı ayaklar sınıra dayandı: {s['up_sat_deg']}°")
if s["touch_deg"] is not None and (s["lost_deg"] is None or s["touch_deg"] < s["lost_deg"] - 0.5):
    print(f"dikkat: ayak dışı bir parça {s['touch_deg']}°'de zemine değdi (bacak katlandı)")

import matplotlib.pyplot as plt
up = [FEET.index(f) for f in ORIENTS[ORIENT][1]]
m_ = a[:, 1] > 0
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(a[m_, 1], a[m_, NN:NN + 4][:, up].mean(1), lw=2, label="yukarı ayaklar")
ax[0].plot(a[m_, 1], np.delete(a[m_, NN:NN + 4], up, 1).mean(1), lw=2, label="aşağı ayaklar")
ax[0].set_xlabel("eğim [°]"); ax[0].set_ylabel("ayak başına yük [% ağırlık]"); ax[0].legend()
ax[1].plot(a[m_, 1], a[m_, TN:TN + 4][:, up].mean(1), lw=2, label="yukarı ayaklar |T|/N")
ax[1].plot(a[m_, 1], np.delete(a[m_, TN:TN + 4], up, 1).mean(1), lw=2, label="aşağı ayaklar |T|/N")
ax[1].axhline(MU, ls="--", color="gray", label="μ (sınır)")
ax[1].set_xlabel("eğim [°]"); ax[1].set_ylabel("|T| / N"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4 · Tablo: bizim koşular

Videodaki altı koşuyu yeniden koşturuyor ve bizim kaydettiğimiz sayılarla (`public/v6/kk-04-v3/data/tilt_go2.json`, MuJoCo 3.17.0) karşılaştırıyor. Farklı bir MuJoCo / Menagerie sürümünde son basamak biraz kayabilir.

In [ ]:
CASES = {"buz μ 0,1": (0.1, 0), "buz + 5 kg": (0.1, 5), "μ 0,5": (0.5, 0), "μ 0,5 + 5 kg": (0.5, 5), "μ 1,25": (1.25, 0), "μ 1,25 + 5 kg": (1.25, 5)}
OURS = {"buz μ 0,1": ("slide", 5.95), "buz + 5 kg": ("slide", 5.87), "μ 0,5": ("tip", 25.63), "μ 0,5 + 5 kg": ("tip", 21.69),
        "μ 1,25": ("tip", 27.49), "μ 1,25 + 5 kg": ("tip", 25.14)}
print(f"{'koşu':<16}{'arctan μ':>10}{'olay':>9}{'açı':>8}   bizimkiyle")
for name, (mu, load) in CASES.items():
    s, _ = run(mu, load)
    got = (s["event"], s["lost_deg"])
    same = "✓ aynı" if got == OURS[name] else f"≠ bizim: {OURS[name]}"
    print(f"{name:<16}{s['theory_deg']:>9}°{s['event']:>9}{s['lost_deg']:>7}°   {same}")

## 5 · Ne gördük?

- **Buzda (μ 0,1)** köpek bir blok gibi davranıyor: 6°'de dört ayağıyla birden kayıyor, teori 5,71°. Sırtına 5 kg koyunca da aynı açı: kaymayı ağırlık değil μ belirliyor.
- **Eğitim aralığında (0,5 ve 1,25)** önce başka bir şey oluyor: zemin eğildikçe ağırlık arka ayaklara biniyor, ön ayakların yükü sıfıra iniyor. Yükü azalan ayağın tutabileceği yanal kuvvet de azalıyor (μ × N), ön ayaklar kayıyor ve köpek arkaya devriliyor. μ'yü 2,5 katına çıkarmak bu duruşta yalnızca ~2° kazandırıyor.

## 6 · Soru

Buzun μ'sü (ayakkabı–buz ≈ 0,1, [OpenStax Tablo 6.1](https://openstax.org/books/university-physics-volume-1/pages/6-2-friction)) eğitim aralığının (0,5–1,25) çok dışında. Bu aralıkta eğitilmiş bir yürüme politikası buzda ne yapar? Tahminini yorumlara yaz.
İpucu: `MU = 0.1` ile `ORIENT = "side"` ve `"down"`'ı dene; düz zeminde (eğim 0) bacakların ayakları dışarı itmesi bile |T|/N ≈ 0,2–0,3 istiyor (grafik 2).